# Embedding and Retrieval Benchmark

This notebook evaluates lexical BM25, dense retrieval, and reciprocal-rank-fusion hybrid retrieval on a fixed corpus. Change one parameter at a time and retain each `benchmark-results.json` output with the experiment notes.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src').exists())
sys.path.insert(0, str(PROJECT_ROOT))

from src.benchmark import (
    Query, build_bm25, chunk_documents, evaluate_rankings, load_documents,
    reciprocal_rank_fusion, search_bm25, search_dense,
)

In [ ]:
CHUNK_SIZE = 80
CHUNK_OVERLAP = 16
TOP_K = 10
K_VALUES = [1, 3, 5, 10]
DENSE_MODELS = [
    'sentence-transformers/all-MiniLM-L6-v2',
    'BAAI/bge-small-en-v1.5',
]

documents = load_documents(PROJECT_ROOT / 'data' / 'sample_corpus.jsonl')
queries = [
    Query('q-bm25', 'How does BM25 rank documents?', ('doc-002',)),
    Query('q-dense', 'What is semantic vector search?', ('doc-003',)),
    Query('q-hybrid', 'How can I combine different retrievers?', ('doc-004',)),
    Query('q-metrics', 'Which metric credits a relevant result near the top?', ('doc-006',)),
    Query('q-chunks', 'Why use overlap between chunks?', ('doc-005',)),
]
chunks = chunk_documents(documents, CHUNK_SIZE, CHUNK_OVERLAP)
len(documents), len(chunks), len(queries)

In [ ]:
records = []

bm25 = build_bm25(chunks)
lexical_rankings, lexical_latency_ms = search_bm25(bm25, chunks, queries, TOP_K)
lexical_metrics = evaluate_rankings(queries, lexical_rankings, K_VALUES)
records.append({
    'method': 'lexical', 'model': 'BM25', 'query_latency_ms': lexical_latency_ms,
    'index_size_bytes': sum(len(chunk.text.encode('utf-8')) for chunk in chunks),
    **lexical_metrics,
})

dense_rankings_by_model = {}
for model_name in DENSE_MODELS:
    rankings, latency_ms, index_size_bytes = search_dense(model_name, chunks, queries, TOP_K)
    dense_rankings_by_model[model_name] = rankings
    records.append({
        'method': 'dense', 'model': model_name, 'query_latency_ms': latency_ms,
        'index_size_bytes': index_size_bytes, **evaluate_rankings(queries, rankings, K_VALUES),
    })

results = pd.DataFrame(records).sort_values(['ndcg@10', 'mrr'], ascending=False)
results

In [ ]:
best_dense_model = results.loc[results['method'].eq('dense'), 'model'].iloc[0]
hybrid_rankings = {
    query.id: reciprocal_rank_fusion([lexical_rankings[query.id], dense_rankings_by_model[best_dense_model][query.id]], TOP_K)
    for query in queries
}
hybrid_metrics = evaluate_rankings(queries, hybrid_rankings, K_VALUES)
hybrid_row = {
    'method': 'hybrid', 'model': f'BM25 + {best_dense_model} (RRF)',
    'query_latency_ms': lexical_latency_ms + results.loc[results['model'].eq(best_dense_model), 'query_latency_ms'].iloc[0],
    'index_size_bytes': results.loc[results['model'].eq(best_dense_model), 'index_size_bytes'].iloc[0],
    **hybrid_metrics,
}
results = pd.concat([results, pd.DataFrame([hybrid_row])], ignore_index=True).sort_values(['ndcg@10', 'mrr'], ascending=False)
results

In [ ]:
output_path = PROJECT_ROOT / 'results' / 'benchmark-results.json'
report_path = PROJECT_ROOT / 'results' / 'tradeoff-report.md'
output_path.write_text(results.to_json(orient='records', indent=2))
best = results.iloc[0]
report_path.write_text(
    '# Retrieval Benchmark Trade-off Report\n\n'
    f'Best configuration on this dataset: **{best["method"]} / {best["model"]}**.\n\n'
    '## Results\n\n' + results.to_markdown(index=False) + '\n\n'
    '## Interpretation\n\n'
    '- BM25 is a low-cost lexical baseline and should remain in every comparison.\n'
    '- Dense retrieval captures semantic matches but adds model download, embedding build, and vector storage costs.\n'
    '- Hybrid reciprocal-rank fusion can improve recall when lexical and dense rankings make different errors.\n'
    '- The fixture corpus is a smoke test only; use versioned domain queries and graded qrels before making product decisions.\n'
)
print(f'Wrote {output_path}')
print(f'Wrote {report_path}')